# Метрики качества регрессии — простыми словами

Учебный ноутбук: как понять, **насколько хорошо** модель предсказывает **числа** (цену, продажи, температуру…).

Разбираем **MSE, RMSE, MAE, MAPE, R², Adjusted R², MSLE, RMSLE** — без страшной теории, с маленькими примерами и кодом.

**Что понадобится**
- Python + Jupyter / VS Code / Colab
- `numpy`, `scikit-learn` (желательно ≥ 1.4 для `root_mean_squared_error`)

Запускайте ячейки **по порядку**.

---

## План

1. [Словарь и общая идея](#dict)
2. [Один игрушечный пример на все метрики](#toy)
3. [MSE — средняя квадратичная ошибка](#mse)
4. [RMSE — корень из MSE](#rmse)
5. [MAE — средняя абсолютная ошибка](#mae)
6. [Сравнение MSE / RMSE / MAE на выбросе](#compare)
7. [MAPE — ошибка в процентах](#mape)
8. [R² — коэффициент детерминации](#r2)
9. [Adjusted R² — штраф за лишние признаки](#adj)
10. [MSLE и RMSLE — через логарифмы](#msle)
11. [Train / Valid / Test: какие метрики где смотреть](#splits)
12. [Шпаргалка и мини-практика](#итог)


<a id="dict"></a>
## 1. Словарь и общая идея

| Термин | Простыми словами |
|--------|------------------|
| **Регрессия** | Модель предсказывает **число** (не класс «да/нет») |
| **$y_i$** | Настоящий (правильный) ответ для объекта $i$ |
| **$\hat{y}_i$** (y-hat) | Предсказание модели |
| **Ошибка** | $y_i - \hat{y}_i$ (может быть + или −) |
| **Метрика качества** | Одно число: «насколько модель хороша / плоха» |
| **Loss (функция потерь)** | То, что модель **минимизирует при обучении** (часто похоже на метрику) |
| **Чем меньше — тем лучше** | MSE, RMSE, MAE, MAPE, MSLE, RMSLE |
| **Чем ближе к 1 — тем лучше** | R² (и обычно Adjusted R²) |
| **Выброс** | Редкое, очень «странное» значение / огромный промах |
| **Базовая модель** | Глупый прогноз «всегда среднее $y$» — точка отсчёта для R² |

### Зачем вообще несколько метрик?

Одна метрика отвечает на **один** вопрос:

| Вопрос | Метрика |
|--------|---------|
| Насколько велика ошибка, сильно ли болят большие промахи? | **MSE / RMSE** |
| Какова типичная ошибка в рублях/метрах, устойчивее к выбросам? | **MAE** |
| На сколько **процентов** в среднем ошибаемся? | **MAPE** |
| Насколько модель лучше, чем «всегда среднее»? | **R²** |
| То же, но со штрафом за лишние признаки (OLS)? | **Adjusted R²** |
| Важны **относительные** ошибки на разных масштабах? | **MSLE / RMSLE** |

> Хорошая практика: смотреть **2–3 метрики сразу**  
> (например, **MAE + RMSE + R²**).


<a id="toy"></a>
## 2. Один игрушечный пример

Три квартиры. Модель чуть ошиблась:

| Квартира | Цена правда $y$, млн ₽ | Предсказание $\hat{y}$ | Ошибка $y-\hat{y}$ |
|----------|--------------------------|--------------------------|----------------------|
| 1 | 5.0 | 5.2 | −0.2 |
| 2 | 6.0 | 5.5 | +0.5 |
| 3 | 7.0 | 7.5 | −0.5 |

Дальше на этих трёх числах **вручную** посчитаем почти все метрики — так формулы перестанут быть «магией».


In [ ]:
import numpy as np

y_true = np.array([5.0, 6.0, 7.0])
y_pred = np.array([5.2, 5.5, 7.5])

errors = y_true - y_pred
print("Правда:      ", y_true)
print("Предсказание:", y_pred)
print("Ошибки:      ", errors)
print("Модули:      ", np.abs(errors))
print("Квадраты:    ", errors ** 2)


<a id="mse"></a>
## 3. MSE — Mean Squared Error

**Средняя квадратичная ошибка.**

### Вопрос, на который отвечает MSE

> «Насколько **в среднем** предсказания отличаются от правды?»  
> Но отличие берётся **в квадрате**.

### Формула

$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2
$$

- $n$ — число объектов  
- $y_i$ — правда  
- $\hat{y}_i$ — предсказание  

### Почему квадрат?

1. **Убирает знак** (−0.5 и +0.5 одинаково «плохи»).  
2. **Сильнее штрафует большие ошибки**: ошибка 2 → вклад 4; ошибка 4 → вклад 16.

Аналогия: опоздать на 2 минуты — неприятно; на 20 — **не в 10 раз**, а **гораздо** хуже, если считать квадрат.

### На наших 3 квартирах

$$
MSE = \frac{(-0.2)^2 + (0.5)^2 + (-0.5)^2}{3} = \frac{0.04 + 0.25 + 0.25}{3} = \frac{0.54}{3} = 0.18
$$

Единицы: если $y$ в **млн ₽**, MSE — в **(млн ₽)²**. Это неудобно для человека (см. RMSE).

### Когда использовать MSE

1. Задача — **регрессия**.  
2. **Большие** ошибки особенно опасны.  
3. Нет огромного количества случайных **выбросов** (или вы их обработали).  
4. Модель учится **градиентными** методами — квадрат математически удобен (часто как **loss**).

### Когда лучше другое

| Вместо MSE | Если… |
|------------|--------|
| **MAE** | Все ошибки «равноценны», много выбросов |
| **RMSE** | Хотите смысл MSE, но в **тех же единицах**, что $y$ |
| **MAPE** | Нужны **проценты** |

### Самое важное про MSE

1. Только для **регрессии**.  
2. Это **средний квадрат** ошибки.  
3. Квадрат: без знака + штраф крупным промахам.  
4. **Чем меньше — тем лучше.**  
5. Минусы: чувствительность к выбросам; единицы «в квадрате».  
6. Часто MSE = **loss при обучении**, а человеку показывают **RMSE/MAE**.


In [ ]:
from sklearn.metrics import mean_squared_error

# Вручную
mse_hand = np.mean((y_true - y_pred) ** 2)
# sklearn
mse_sk = mean_squared_error(y_true, y_pred)

print(f"MSE вручную:  {mse_hand:.4f}")
print(f"MSE sklearn:  {mse_sk:.4f}")
print("Совпадает?", np.isclose(mse_hand, mse_sk))

# Типичный вызов после model.predict:
# y_pred = model.predict(X_test)
# mse = mean_squared_error(y_test, y_pred)


<a id="rmse"></a>
## 4. RMSE — Root Mean Squared Error

**Корень** из MSE.

### Зачем

У MSE недостаток: если $y$ в рублях, MSE — в **рублях²**.  
Трудно сказать заказчику: «ошибка 250 000 000 000».

Берём корень:

$$
RMSE = \sqrt{MSE}
$$

Теперь можно сказать ближе к человеческому языку:  
«Модель в среднем ошибается **примерно на … рублей**»  
(с оговоркой: это не то же самое, что MAE — RMSE всё ещё сильнее бьёт по крупным ошибкам).

### На нашем примере

$$
RMSE = \sqrt{0.18} \approx 0.424\ \text{млн ₽}
$$

### Плюсы / минусы

| | |
|--|--|
| **+** | Те же единицы, что $y$ |
| **+** | Как MSE, сильно штрафует большие ошибки |
| **−** | Так же чувствителен к выбросам |

### Что запомнить

1. **MSE** — удобнее для **обучения** (математика, градиенты).  
2. **RMSE** — удобнее для **человека**.  
3. Это **одна и та же информация**, RMSE = «MSE, возвращённый в исходные единицы».

### Когда использовать

Почти всегда, когда уже думаете про MSE и хотите **понятную** величину ошибки.


In [ ]:
from sklearn.metrics import root_mean_squared_error, mean_squared_error

rmse_hand = np.sqrt(mean_squared_error(y_true, y_pred))
rmse_sk = root_mean_squared_error(y_true, y_pred)

print(f"RMSE вручную (sqrt MSE): {rmse_hand:.4f}")
print(f"RMSE sklearn:            {rmse_sk:.4f}")

# Если старая версия sklearn без root_mean_squared_error:
# rmse = np.sqrt(mean_squared_error(y_true, y_pred))


<a id="mae"></a>
## 5. MAE — Mean Absolute Error

**Средняя абсолютная ошибка.**

### Вопрос

> На **сколько единиц в среднем** предсказания отличаются от правды?

Если MAE = 50 000 ₽ → «в среднем ошибаемся примерно на 50 000 ₽».

### Формула

$$
MAE = \frac{1}{n}\sum_{i=1}^{n}\lvert y_i - \hat{y}_i\rvert
$$

### На нашем примере

$$
MAE = \frac{0.2 + 0.5 + 0.5}{3} = \frac{1.2}{3} = 0.4\ \text{млн ₽}
$$

Заметьте: RMSE ≈ 0.42, MAE = 0.4 — близко, потому что **нет огромного выброса**.  
Если появится огромный промах, RMSE «улетит» сильнее MAE (следующий раздел).

### Главное отличие от MSE

| | MAE | MSE / RMSE |
|--|-----|------------|
| Как берём ошибку | **модуль** $|e|$ | **квадрат** $e^2$ |
| Большие ошибки | линейно (×2 ошибка ≈ ×2 вклад) | **нелинейно** сильнее |
| Выбросы | устойчивее | чувствительнее |
| Единицы | как у $y$ | MSE — $y^2$; RMSE — как у $y$ |

### Когда применять MAE

1. Нужна **понятная** средняя ошибка в исходных единицах.  
2. В данных есть **выбросы**.  
3. Одна гигантская ошибка **не должна** «убить» всю метрику.  
4. Цена ошибки растёт **примерно линейно** (промах на 20 ₽ ≈ вдвое хуже, чем на 10 ₽).

### Плюсы / минусы

**Плюсы:** интерпретация; те же единицы; устойчивее к выбросам; ошибки пропорциональны размеру.  

**Минусы:** не «кричит» о редких катастрофических промахах; как loss модуль менее удобен для оптимизации (излом в нуле).

### Главное

> MAE — средний **размер** ошибки без знака.  
> **MAE** — когда нужна понятная и относительно устойчивая оценка.  
> **MSE/RMSE** — когда крупные ошибки нужно наказывать **сильнее**.


In [ ]:
from sklearn.metrics import mean_absolute_error

mae_hand = np.mean(np.abs(y_true - y_pred))
mae_sk = mean_absolute_error(y_true, y_pred)

print(f"MAE вручную: {mae_hand:.4f}")
print(f"MAE sklearn: {mae_sk:.4f}")


<a id="compare"></a>
## 6. MSE / RMSE / MAE: один выброс меняет картину

Добавим **четвёртую** «квартиру», где модель сильно промахнулась (или в данных выброс).


In [ ]:
# Те же 3 точки + один огромный промах
y2 = np.array([5.0, 6.0, 7.0, 8.0])
p2 = np.array([5.2, 5.5, 7.5, 20.0])  # предсказали 20 вместо 8!

def report(name, yt, yp):
    mae = mean_absolute_error(yt, yp)
    mse = mean_squared_error(yt, yp)
    rmse = root_mean_squared_error(yt, yp)
    print(f"{name}")
    print(f"  MAE  = {mae:.3f}")
    print(f"  MSE  = {mse:.3f}")
    print(f"  RMSE = {rmse:.3f}")
    print(f"  RMSE / MAE ≈ {rmse / mae:.2f}  (чем больше, тем сильнее «хвост» больших ошибок)")

report("Без выброса (3 точки):", y_true, y_pred)
print()
report("С выбросом (4 точки):", y2, p2)
print()
print("Вывод: один крупный промах сильнее раздувает MSE/RMSE, чем MAE.")


**Как читать RMSE / MAE**

- Если RMSE **заметно больше** MAE — в ошибках есть **тяжёлый хвост** (редкие крупные промахи).  
- Если RMSE ≈ MAE — ошибки более «ровные».

Это не строгий тест, но полезная быстрая диагностика.


<a id="mape"></a>
## 7. MAPE — Mean Absolute Percentage Error

**Средняя абсолютная процентная ошибка.**

### Вопрос

> На сколько **процентов** в среднем ошибаемся?

### Формула

$$
MAPE = \frac{1}{n}\sum_{i=1}^{n}\left\lvert\frac{y_i - \hat{y}_i}{y_i}\right\rvert
$$

Часто умножают на 100% → «ошибка 10%».

### MAE vs MAPE

| | MAE | MAPE |
|--|-----|------|
| Единицы | рубли, метры… | **доли / проценты** |
| Ошибка 10 000 ₽ на квартире 100 000 ₽ | 10 000 | **10%** |
| Ошибка 10 000 ₽ на квартире 10 000 000 ₽ | 10 000 | **0.1%** |

MAE в обоих случаях **одинаковая**, MAPE — **разная**.  
MAPE удобна, когда объекты **разного масштаба**.

### Когда применять

1. Важна ошибка **в процентах**.  
2. $y$ **положительные**.  
3. $y$ **не ноль** и не «почти ноль».  
4. Нужно сравнивать качество на разных масштабах.

Часто: продажи, спрос, выручка, цены, потребление ресурсов.

### Опасности

**1. $y = 0$** → деление на ноль.  

**2. $y$ около нуля** → крошечная абсолютная ошибка даёт **огромный** процент:

- правда = 1, прогноз = 3 → $|1-3|/1 = 200\%$.

**3. Несимметричность**

| Правда | Прогноз | MAPE-вклад |
|--------|---------|------------|
| 100 | 200 | $|100-200|/100$ = **100%** |
| 200 | 100 | $|200-100|/200$ = **50%** |

Числа просто поменялись местами, а «штраф» разный.

### Плюсы / минусы (кратко)

**+** понятна бизнесу; не привязана к единицам; сравнение масштабов.  
**−** нули/около нуля; асимметрия; иногда «врёт» на малых $y$.


In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

# Квартиры разного масштаба, одинаковая абсолютная ошибка 0.5
y_small = np.array([5.0, 6.0])      # «дешёвые»
p_small = np.array([5.5, 6.5])      # ошибка +0.5

y_big = np.array([50.0, 60.0])      # «дорогие»
p_big = np.array([50.5, 60.5])      # та же ошибка +0.5

print("MAE small:", mean_absolute_error(y_small, p_small))
print("MAE big:  ", mean_absolute_error(y_big, p_big))
print("→ MAE одинаковая")
print()
print("MAPE small:", mean_absolute_percentage_error(y_small, p_small),
      f"→ {100*mean_absolute_percentage_error(y_small, p_small):.1f}%")
print("MAPE big:  ", mean_absolute_percentage_error(y_big, p_big),
      f"→ {100*mean_absolute_percentage_error(y_big, p_big):.1f}%")
print("→ MAPE разная: на дешёвых относительная ошибка больше")
print()

# Асимметрия
print("MAPE 100→200:", mean_absolute_percentage_error([100], [200]))
print("MAPE 200→100:", mean_absolute_percentage_error([200], [100]))

# Наш toy
print()
print("MAPE на 3 квартирах:", mean_absolute_percentage_error(y_true, y_pred),
      f"({100*mean_absolute_percentage_error(y_true, y_pred):.2f}%)")


<a id="r2"></a>
## 8. R² — Coefficient of Determination

**Коэффициент детерминации.**

### Вопрос (не «на сколько ошиблись», а…)

> Какую **долю изменчивости** $y$ модель **объяснила**?  
> Насколько она **лучше**, чем тупо всегда предсказывать **среднее** $\bar{y}$?

Пример: **R² = 0.85** ≈ «модель объяснила **85%** вариации данных» (грубо говоря).

### Формула

$$
R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}
$$

- числитель — **RSS** (сумма квадратов ошибок модели);  
- знаменатель — **TSS** (сумма квадратов отклонений от среднего — ошибка «глупой» модели).

### Два варианта предсказания цены

1. **Без ML:** всегда говорить «средняя цена = 10 млн».  
2. **С ML:** нормальная модель.

R² спрашивает: *насколько вариант 2 лучше варианта 1 по сумме квадратов ошибок?*

### Как читать

| R² | Смысл |
|----|--------|
| **1** | Идеально: все предсказания совпали |
| **0** | Не лучше, чем всегда предсказывать среднее |
| **&lt; 0** | **Хуже** среднего (модель вредная / плохо настроена / на test развалилась) |

### MSE vs R²

| | MSE / MAE / RMSE | R² |
|--|------------------|-----|
| Вопрос | **Насколько велика** ошибка? | **Насколько хорошо** объясняем данные? |
| Единицы | да (кроме безразмерных %) | **без** единиц (доля) |
| Вместе | Часто выводят **и** ошибку, **и** R² | |

### Плюсы / минусы

**+** понятная шкала; удобно сравнивать модели **на одной задаче**; якорь — baseline «среднее».  

**−** не говорит «ошибка 50 тыс. ₽»; может быть высоким при редких больших промахах; **нельзя** честно сравнивать R² между **совершенно разными** датасетами/задачами.

### Главное

> R² — не размер ошибки, а **качество объяснения** относительно среднего.  
> На практике: **MAE/RMSE + R²** — хорошая связка.


In [ ]:
from sklearn.metrics import r2_score

# Вручную на toy
ss_res = np.sum((y_true - y_pred) ** 2)          # RSS — ошибки модели
ss_tot = np.sum((y_true - y_true.mean()) ** 2)    # TSS — ошибки «всегда среднее»
r2_hand = 1 - ss_res / ss_tot
r2_sk = r2_score(y_true, y_pred)

print(f"Среднее y = {y_true.mean():.3f}")
print(f"RSS (модель) = {ss_res:.4f}")
print(f"TSS (среднее) = {ss_tot:.4f}")
print(f"R² вручную = {r2_hand:.4f}")
print(f"R² sklearn = {r2_sk:.4f}")
print()

# R² = 0: предсказываем всегда среднее
y_mean_pred = np.full_like(y_true, y_true.mean())
print("R² если всегда среднее:", r2_score(y_true, y_mean_pred))

# R² < 0: совсем плохие предсказания
y_bad = np.array([10.0, 10.0, 10.0])
print("R² плохой модели:", r2_score(y_true, y_bad))


### Мини-демо: «хороший» R² не отменяет RMSE

Иногда R² выглядит прилично, а в рублях ошибка всё ещё большая — зависит от **разброса** $y$.  
Поэтому одной R² мало для бизнеса: «R²=0.9» не равно «ошибка маленькая в ₽».


In [ ]:
# Два датасета с похожим R², разным масштабом ошибки
np.random.seed(0)
n = 40

def make_case(scale, noise):
    x = np.linspace(0, 1, n)
    y = scale * (2 * x + 1) + np.random.normal(0, noise, n)
    # простая «модель»: почти правда + чуть шума в предсказании
    pred = scale * (2 * x + 1) + np.random.normal(0, noise * 0.5, n)
    return y, pred

for name, scale, noise in [("малый масштаб", 1.0, 0.3), ("крупный масштаб", 100.0, 30.0)]:
    y, p = make_case(scale, noise)
    print(f"{name}: R²={r2_score(y, p):.3f}, RMSE={root_mean_squared_error(y, p):.2f}, "
          f"MAE={mean_absolute_error(y, p):.2f}")


<a id="adj"></a>
## 9. Adjusted R² — скорректированный R²

### Смысл

**R² со штрафом за число признаков.**

Обычный R² (на **train**, для **линейной регрессии OLS**) почти всегда **не падает**, если добавить признаки — даже **бесполезные**.  
Adjusted R² спрашивает: «Признаков стало больше — модель **реально** лучше?»

### Формула

$$
R^2_{adj} = 1 - (1 - R^2)\frac{n - 1}{n - p - 1}
$$

- $n$ — число объектов  
- $p$ — число признаков  

### Когда применять

Сравниваете модели **с разным числом признаков** (классическая линейная регрессия / статистика).

### Плюсы / минусы

**+** не поощряет мусорные признаки; лучше для сравнения разной сложности.  
**−** в «классическом ML» реже, чем R²; в основном про линейные модели.  
В **sklearn отдельной функции нет** — считают вручную.

### ВАЖНЫЙ МОМЕНТ (про рост R²)

Фраза «R² почти всегда растёт при добавлении признаков» относится к **OLS линейной регрессии** и к **train**.

Почему: новая модель **содержит** старую как частный случай ($b_{\text{новый}} = 0$).  
Значит сумма квадратов ошибок на train **не может вырасти** → train R² **не уменьшается**.

| | Train | Test |
|--|-------|------|
| OLS + новый признак | R² **≥** старого | R² **может упасть** (переобучение) |

Именно поэтому придумали Adjusted R² и почему всё равно смотрят **test/valid**, а не только train R².


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

def adjusted_r2(r2, n, p):
    # R2_adj: n — число объектов, p — число признаков
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

np.random.seed(42)
# 80 объектов, сначала 3 полезных признака
X, y = make_regression(n_samples=80, n_features=3, n_informative=3, noise=10.0, random_state=42)

# Добавим 5 ПОЛНОСТЬЮ случайных (бесполезных) признаков
noise_feats = np.random.randn(80, 5)
X_full = np.hstack([X, noise_feats])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(X_full, y, test_size=0.25, random_state=0)

def eval_lr(Xtr, Xte, ytr, yte, label):
    m = LinearRegression().fit(Xtr, ytr)
    r2_tr = r2_score(ytr, m.predict(Xtr))
    r2_te = r2_score(yte, m.predict(Xte))
    n, p = Xtr.shape
    print(f"{label} (p={p})")
    print(f"  Train R²={r2_tr:.4f}, Train Adj.R²={adjusted_r2(r2_tr, n, p):.4f}")
    print(f"  Test  R²={r2_te:.4f}, Test  Adj.R²={adjusted_r2(r2_te, len(yte), p):.4f}")

eval_lr(X_tr, X_te, y_tr, y_te, "Только 3 полезных признака")
print()
eval_lr(Xf_tr, Xf_te, yf_tr, yf_te, "3 полезных + 5 мусорных")
print()
print("Смотрите: на TRAIN обычный R² часто подрастает/держится,")
print("Adj.R² и TEST R² честнее показывают, что мусор не помог.")


<a id="msle"></a>
## 10. MSLE и RMSLE — ошибка в «логарифмическом» мире

### MSLE — Mean Squared Logarithmic Error

Считает квадрат разницы **не между $y$**, а между **логарифмами**:

$$
MSLE = \frac{1}{n}\sum_{i=1}^{n}\bigl(\log(1+y_i) - \log(1+\hat{y}_i)\bigr)^2
$$

(`log(1+·)` — чтобы ноль не ломал логарифм; значения должны быть **≥ 0**.)

### Смысл

Сравнивает скорее **относительные** ошибки.

Пример:

| Правда | Прогноз | Абсолютная ошибка | Относительно |
|--------|---------|-------------------|--------------|
| 100 | 110 | 10 | ~10% |
| 1000 | 1100 | 100 | ~10% |

Для **MSE** второй промах **гораздо** тяжелее ($100^2$ vs $10^2$).  
Для **MSLE** они **похожи** (оба ~10% в лог-шкале).

### Когда применять

- важны **относительные** ошибки;  
- $y$ различаются в **десятки/сотни** раз;  
- нет **отрицательных** значений.

### RMSLE

$$
RMSLE = \sqrt{MSLE}
$$

Как RMSE к MSE: тот же смысл, чуть удобнее смотреть «корень».

### Плюсы / минусы (оба)

**+** относительность; менее бьют «просто очень большие» $y$.  
**−** нельзя на отрицательных; интерпретация сложнее, чем у MAE/RMSE.

### Главное

> MSLE/RMSLE ≈ RMSE-идея, но **по логарифмам** (относительные ошибки).


In [ ]:
from sklearn.metrics import mean_squared_log_error, root_mean_squared_log_error

# Два случая с ~10% относительной ошибкой
y_a, p_a = np.array([100.0]), np.array([110.0])
y_b, p_b = np.array([1000.0]), np.array([1100.0])

print("Случай 100→110:")
print(f"  MSE   = {mean_squared_error(y_a, p_a):.1f}")
print(f"  MSLE  = {mean_squared_log_error(y_a, p_a):.6f}")
print(f"  RMSLE = {root_mean_squared_log_error(y_a, p_a):.6f}")

print("Случай 1000→1100:")
print(f"  MSE   = {mean_squared_error(y_b, p_b):.1f}  ← в 100 раз больше по квадрату (10² vs 100²)")
print(f"  MSLE  = {mean_squared_log_error(y_b, p_b):.6f}  ← почти как у 100→110")
print(f"  RMSLE = {root_mean_squared_log_error(y_b, p_b):.6f}")

print()
print("MSE видит разный «вес», MSLE/RMSLE — похожую относительную ошибку.")


<a id="splits"></a>
## 11. Train / Valid / Test: где смотреть метрики

Связь с ноутбуком про **валидацию**:

| Метрика на… | Зачем |
|-------------|--------|
| **Train** | Диагностика обучения: учится ли модель? недообучение / переобучение |
| **Validation** (или CV) | **Выбор** модели и гиперпараметров |
| **Test** | **Честная** финальная оценка на «невиданных» данных |

### Разрыв (gap)

$$
\text{gap} \approx \text{MSE}_{\text{test}} - \text{MSE}_{\text{train}}
$$
(или RMSE/MAE — та же идея)

- **Большой gap** (train намного лучше test) → похоже на **переобучение** («зазубрила» train).  
- **Оба плохие** → **недообучение** (модель слишком простая / мало данных / плохие признаки).  

### Главное правило

1. **Train MSE** — не для выбора «лучшей» модели между кандидатами.  
2. **Validation MSE** — для настройки и сравнения **в процессе**.  
3. **Test MSE** — финальный отчёт.  

Сравнивают модели по **valid/test**, не по train.

> На практике MSE часто минимизируют как **loss**,  
> а в отчёте показывают **RMSE/MAE + R²**.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

np.random.seed(1)
X = np.linspace(0, 1, 50).reshape(-1, 1)
y = np.sin(2 * np.pi * X.ravel()) + np.random.normal(0, 0.15, size=50)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

print(f"{'degree':>6} | {'RMSE train':>10} | {'RMSE test':>10} | {'gap':>8} | {'R² test':>8}")
print("-" * 55)
for d in [1, 3, 8, 15]:
    model = make_pipeline(PolynomialFeatures(d, include_bias=False), LinearRegression())
    model.fit(X_tr, y_tr)
    tr = root_mean_squared_error(y_tr, model.predict(X_tr))
    te = root_mean_squared_error(y_te, model.predict(X_te))
    r2 = r2_score(y_te, model.predict(X_te))
    print(f"{d:6d} | {tr:10.4f} | {te:10.4f} | {te-tr:8.4f} | {r2:8.3f}")

print()
print("Маленькая degree: train и test оба так себе → недообучение.")
print("Очень большая degree: train отличный, test хуже, gap большой → переобучение.")
print("Выбор degree — по valid/test (или CV), не по train.")


<a id="итог"></a>
## 12. Самое главное + шпаргалка

### Одной таблицей

| Метрика | Идея | Единицы | К выбросам | sklearn |
|---------|------|---------|------------|---------|
| **MSE** | средний квадрат ошибки | $y^2$ | очень чувствительна | `mean_squared_error` |
| **RMSE** | $\sqrt{MSE}$ | как $y$ | как MSE | `root_mean_squared_error` |
| **MAE** | средний модуль ошибки | как $y$ | устойчивее | `mean_absolute_error` |
| **MAPE** | средний **%** ошибки | % / доля | опасна при $y\approx 0$ | `mean_absolute_percentage_error` |
| **R²** | лучше ли среднего? | безразмерная | не заменяет RMSE | `r2_score` |
| **Adj. R²** | R² − штраф за $p$ | безразмерная | для OLS / разного $p$ | вручную |
| **MSLE** | MSE по $\log(1+y)$ | лог-шкала | относительные ошибки | `mean_squared_log_error` |
| **RMSLE** | $\sqrt{MSLE}$ | лог-шкала | то же | `root_mean_squared_log_error` |

### Что выбрать? (старт)

| Ситуация | Старт |
|----------|--------|
| Нужна loss / обучение | **MSE** (часто внутри модели) |
| Объяснить ошибку в ₽ | **MAE** и/или **RMSE** |
| Много выбросов, линейная цена ошибки | **MAE** |
| Крупные промахи критичны | **MSE / RMSE** |
| Важны проценты, $y > 0$ | **MAPE** (осторожно у нуля) |
| Разные масштабы $y$, относительность | **MSLE / RMSLE** или MAPE |
| «Насколько модель вообще ок» | **R²** (+ всегда ещё MAE/RMSE) |
| Сравниваем число признаков (OLS) | **Adjusted R²** + test-метрики |

### Связка «хороший отчёт»

```text
RMSE (или MAE)  — размер ошибки
R²              — доля объяснённой вариации
(+ gap train/test — переобучение)
```

### Мини-глоссарий

| Слово | Смысл |
|-------|--------|
| **RSS** | сумма квадратов ошибок модели |
| **TSS** | сумма квадратов отклонений от $\bar{y}$ |
| **Loss** | что минимизируем при `fit` |
| **Gap** | разрыв train vs test метрик |
| **Baseline** | «всегда среднее» для R² |


### Мини-практика

На синтетических данных:

1. Обучите `LinearRegression`.  
2. Посчитайте на **test**: MAE, RMSE, MSE, R², MAPE.  
3. Напечатайте аккуратную таблицу.  
4. (Опционально) Добавьте один искусственный выброс в предсказания и посмотрите, что сильнее выросло — MAE или RMSE.


In [ ]:
# ===== Мини-практика / эталонный шаблон =====
import numpy as np
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)

np.random.seed(42)
X, y = make_regression(n_samples=200, n_features=4, noise=12.0, random_state=42)
# Сдвинем y в положительную область, чтобы MAPE был осмысленным
y = y - y.min() + 10.0

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

metrics = {
    "MAE": mean_absolute_error(y_test, y_pred),
    "MSE": mean_squared_error(y_test, y_pred),
    "RMSE": root_mean_squared_error(y_test, y_pred),
    "MAPE": mean_absolute_percentage_error(y_test, y_pred),
    "R2": r2_score(y_test, y_pred),
}

print("Метрики на TEST:")
for k, v in metrics.items():
    if k == "MAPE":
        print(f"  {k:5s} = {v:.4f}  ({100*v:.2f}%)")
    elif k == "R2":
        print(f"  {k:5s} = {v:.4f}")
    else:
        print(f"  {k:5s} = {v:.4f}")

print(f"\nRMSE/MAE ≈ {metrics['RMSE']/metrics['MAE']:.3f}")

# Опционально: выброс
y_test_out = y_test.copy()
y_pred_out = y_pred.copy()
y_pred_out[0] = y_pred_out[0] + 50 * np.std(y_test)  # огромный промах на 1 объекте

print("\nПосле искусственного огромного промаха на 1 объекте:")
print(f"  MAE  {mean_absolute_error(y_test_out, y_pred_out):.2f}  (было {metrics['MAE']:.2f})")
print(f"  RMSE {root_mean_squared_error(y_test_out, y_pred_out):.2f}  (было {metrics['RMSE']:.2f})")
print("  → RMSE обычно растёт сильнее относительно «типичной» картины ошибок.")


## Что делать дальше

1. Прогоните все ячейки и сверьте ручной расчёт toy-примера со `sklearn`.  
2. В своих проектах договоритесь о **главной** метрике (часто MAE или RMSE) и 1–2 дополнительных.  
3. Свяжите с ноутбуком **«Валидация…»**: метрики на CV/valid для выбора, test — один раз.  
4. Помните: метрика должна отражать **бизнес-цену ошибки**, а не только «привычку из учебника».

### Главная мысль занятия

> Метрики регрессии отвечают на разные вопросы:  
> **насколько ошиблись**, **на сколько процентов**, **насколько лучше среднего**.  
> Одной цифры мало — смотрите набор метрик и **train vs test**.

Удачи!
